# Trends in last few months : 

The main trends I’m seeing are:

1) Agents are moving into real workflows, not just chat.
Companies are positioning agents to handle coding, operations, customer support, finance, and internal work streams. OpenAI’s agent-focused products and Google’s agent-first platform messaging both point in the same direction: agents are being treated as workflow workers, not just assistants. Reuters also reported Google, Oracle, and Meta pushing agentic products into enterprise and consumer use cases.

2) Standardization is becoming a big deal.
A lot of the momentum is around protocols that help agents connect to tools and to each other. MCP is now a core standard for connecting AI apps to external systems, and A2A is being used for agent-to-agent communication and discovery. Google’s recent protocol guides and Anthropic’s MCP docs show the ecosystem is trying to move away from one-off integrations toward reusable plumbing.

3) Security, identity, and governance are getting much more attention.
As agents gain tool access, the risk surface grows. NIST released work on software and AI agent identity and authorization, and Reuters has reported legal and operational concerns around agentic AI’s higher autonomy. In practice, this means more human approval steps, tighter permissions, logging, and controls against prompt injection and misuse.

4) Coding agents are one of the hottest battlegrounds.
OpenAI’s Codex positioning, Google’s agent tooling, and broader industry coverage all show that software development is one of the strongest early commercial uses for agentic AI. The reason is simple: coding is measurable, tool-rich, and naturally broken into steps, which makes it a good fit for agent loops.

5) Enterprises are becoming more realistic about ROI.
There is still a lot of hype, but the market is also maturing. Reuters reported concerns that many agentic projects may be scrapped if they do not show real business value, while recent enterprise announcements from Google, Oracle, Vodafone, Capgemini, and others show the move from experimentation to deployment.

6) Multi-agent and “agentic app” designs are rising.
Instead of one giant agent doing everything, many teams are splitting tasks across specialized agents and wrapping them inside business applications. Google’s A2A materials and Oracle’s “agentic apps” direction both reflect that modular pattern.

In [107]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


True

In [108]:
def show(text):
    try:
        Console().print(text)
    except Exception as e:
        print(text)     

In [109]:
openai = OpenAI()

In [110]:
todos: list[str] = []
completed: list[bool] = []

In [111]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: [red]{todo}[/red]\n"
    show(result)        
    return result

In [112]:
get_todo_report()

''

In [113]:
def create_todos(description: list[str]) -> str:
    todos.extend(description)
    completed.extend([False] * len(description))
    return get_todo_report()

In [114]:
def mark_complete(index: int , completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index -1] = True
    else:
        return "No todo found with that index."
    Console().print(f"Todo #{index} marked as complete. Notes: {completion_notes}")
    return get_todo_report()

In [115]:
todos, completed = [] , []

create_todos(["Buy groceries", "Clean the house", "Finish the project"])

Todo #1: Buy groceries
Todo #2: Clean the house
Todo #3: Finish the project

'Todo #1: [red]Buy groceries[/red]\nTodo #2: [red]Clean the house[/red]\nTodo #3: [red]Finish the project[/red]\n'

In [116]:
mark_complete(2, "House is now clean.")

Todo #2 marked as complete. Notes: House is now clean.

Todo #1: Buy groceries
Todo #2: Clean the house
Todo #3: Finish the project

'Todo #1: [red]Buy groceries[/red]\nTodo #2: [green][strike]Clean the house[/strike][/green]\nTodo #3: [red]Finish the project[/red]\n'

In [117]:
create_todo_json = {
    "name" : "create_todos",
    "description" : "Creates new todos based on the provided descriptions.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "description" : {
                "type" : "array",
                "items" : {
                    "type" : "string"
                },
                "description" : "A list of todo descriptions to be added."
            }
        },
        "required" : ["description"],
        "additionalProperties" : False

    }
}

In [118]:
mark_complete_json = {
    "name" : "mark_complete",
    "description" : "Marks a specified todo as complete and adds completion notes.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "index" : {
                "type" : "integer",
                "description" : "The index of the todo to be marked as complete (1-based index)."
            },
            "completion_notes" : {
                "type" : "string",
                "description" : "Notes or comments about the completion of the todo."
            }
        },
        "required" : ["index", "completion_notes"],
        "additionalProperties" : False
    }   
}

In [119]:
tools = [{"type" : "function", "function" : create_todo_json},
         {"type" : "function", "function" : mark_complete_json}]

In [120]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)   # ← add json.loads()
        print(f"Tool call: {tool_name} with arguments {arguments}")
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else f"No tool found with name {tool_name}"
        results.append({"role" : "tool", "content" : json.dumps(result) , "tool_call_id" : tool_call.id})
    return results

In [121]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o", tools=tools, tool_choice="auto", messages=messages)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [124]:
system_message = """
You are given a problem to solve using your todo tools. Follow these steps strictly:

Use create_todos to plan a list of steps for the problem.
After creating the todos, immediately simulate completing each step one by one — call mark_complete for every single todo in order, adding relevant completion notes for each.
Do not stop until ALL todos have been marked as complete.
Once all steps are complete, provide a final summary answering the original question.
"""
user_message = """A train leaves Boston at 2 PM heading to New York. It is expected to arrive at 5 PM. Create a todo list of steps to track the progress of this train journey, then mark each step as complete as the train passes through each milestone. When does the train arrive?"""

messages = [{"role" : "system", "content" : system_message},
            {"role" : "user", "content" : user_message}]

In [125]:
todos ,completed = [] , []
loop(messages)

Tool call: create_todos with arguments {'description': ['Train departs from Boston at 2 PM.', 'Train passes through Providence.', 'Train passes through New Haven.', 'Train passes through Stamford.', 'Train arrives in New York at 5 PM.']}


Todo #1: Train departs from Boston at 2 PM.
Todo #2: Train passes through Providence.
Todo #3: Train passes through New Haven.
Todo #4: Train passes through Stamford.
Todo #5: Train arrives in New York at 5 PM.

Tool call: mark_complete with arguments {'index': 1, 'completion_notes': 'The train departed from Boston on time at 2 PM.'}


Todo #1 marked as complete. Notes: The train departed from Boston on time at 2 PM.

Todo #1: Train departs from Boston at 2 PM.
Todo #2: Train passes through Providence.
Todo #3: Train passes through New Haven.
Todo #4: Train passes through Stamford.
Todo #5: Train arrives in New York at 5 PM.

...

Todo #1: Train departs from Boston at 2 PM.
Todo #2: Train passes through Providence.
Todo #3: Train passes through New Haven.
Todo #4: Train passes through Stamford.
Todo #5: Train arrives in New York at 5 PM.

Tool call: mark_complete with arguments {'index': 5, 'completion_notes': 'The train arrived in New York at the planned time of 5 PM.'}


Todo #5 marked as complete. Notes: The train arrived in New York at the planned time of 5 PM.

Todo #1: Train departs from Boston at 2 PM.
Todo #2: Train passes through Providence.
Todo #3: Train passes through New Haven.
Todo #4: Train passes through Stamford.
Todo #5: Train arrives in New York at 5 PM.

The train departed from Boston at 2 PM, passed through Providence, New Haven, and Stamford, and successfully 
arrived in New York at the planned time of 5 PM.